# Tools: the model chooses the function

Two plain Python functions, one `tools` schema, and a model that decides which one to call.

1. The two functions
2. The tool schema
3. One call, in detail
4. The loop

The model never sees this code. The only things it knows about the tools are the `name`, `description` and `parameters` we pass in `tools`. OpenRouter forwards `tools` to the provider, so the model behind `MODEL` has to support tool calling.


# 1. The two functions

First create the database (from the repo root):

```
python MLAppDev/sprint_3/seed.py
```

It writes `data/app.db` with a `users` table that has a `city` column.


In [55]:
import json
import os
import sqlite3
from pathlib import Path

import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    from google.colab import userdata      # Colab Secrets
    API_KEY = userdata.get("OPENROUTER_API_KEY")

BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "google/gemini-2.5-flash-lite"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

DB_PATH = Path("../data/app.db")     # run the notebook from the repo root
if not DB_PATH.exists():
    raise RuntimeError("Database not found. Run `python MLAppDev/sprint_3/seed.py` first.")


In [11]:
def get_city(user_id: int) -> dict:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        row = conn.execute(
            "SELECT user_id, user_name, country, city FROM users WHERE user_id = ?",
            (user_id,),
        ).fetchone()
    finally:
        conn.close()
    return dict(row) if row else {"error": f"user {user_id} not found"}


In [13]:
get_city(3)


{'user_id': 3,
 'user_name': 'Priya Nair',
 'country': 'India',
 'city': 'Bengaluru'}

In [14]:
def get_weather(city: str) -> dict:
    response = requests.get(f"https://wttr.in/{city}?format=%t", timeout=20)
    return {"city": city, "temperature": response.text.strip()}


In [15]:
get_weather("Cozumel")


{'city': 'Cozumel', 'temperature': '+25°C'}

# 2. The tool schema

`tools` is a list of function descriptions. Each one has three parts:

- `name`: what the model returns when it picks the tool. It has to match our Python function.
- `description`: the only text the model reads to decide when to use the tool. This is where we say that `get_city` needs a `user_id`.
- `parameters`: a JSON Schema for the arguments, with `required` for the ones that cannot be missing.

`tool_choice="auto"` lets the model decide. `"required"` would force some tool, and a named function would force that one.


In [50]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_city",
            "description": (
                "Look up the city of a user in the database. Takes the numeric user_id. If none is given ask for it. Dont ask for city as for user id "
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "integer", "description": "The numeric user id"}
                },
                "required": ["user_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Current temperature in Celsius for a city, from the weather service. Dont ask for it directly, ask for the user Id and then from there get the city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example Berlin"}
                },
                "required": ["city"],
            },
        },
    },
]


# 3. One call, in detail

One question, one request. The model can answer by itself, or it can ask us to run a function.


In [8]:
messages = [{"role": "user", "content": "Whats up bro?"}]

completion = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=TOOLS,
    tool_choice="auto",
    temperature=0,
)

choice = completion.choices[0]
print("finish_reason:", choice.finish_reason)
print("content:", choice.message.content)
print("tool_calls:", choice.message.tool_calls)
print("usage:", completion.usage)


finish_reason: stop
content: Not much, hope you're doing well! Anything I can help you with?
tool_calls: None
usage: CompletionUsage(completion_tokens=17, prompt_tokens=162, total_tokens=179, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None, video_tokens=0), cost=2.3e-05, is_byok=False, cost_details={'upstream_inference_cost': 2.3e-05, 'upstream_inference_prompt_cost': 1.62e-05, 'upstream_inference_completions_cost': 6.8e-06})


In [33]:
completion.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='tool_get_weather_8cPJcPz4AZM68S1fDLuT', function=Function(arguments='{"city":"Paris"}', name='get_weather'), type='function', index=0)]

`finish_reason` is why the model stopped:

- `"tool_calls"`: it wants a function. `content` is `None` and `message.tool_calls` holds the request.
- `"stop"`: it answered by itself.

Each call has an `id` (we send it back), `function.name`, and `function.arguments`, which is a JSON **string**, not a dict.

The loop repeats while there are tool calls, so `finish_reason` is the stop condition.


In [34]:
call = choice.message.tool_calls[0]
args = json.loads(call.function.arguments)

print("name:", call.function.name)
print("args:", args)

result = get_weather(**args)
result


name: get_weather
args: {'city': 'Paris'}


{'city': 'Paris', 'temperature': '+10°C'}

In [9]:
get_weather(city= args["city"])

NameError: name 'args' is not defined

In [35]:
messages.append(choice.message)
messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

completion = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=TOOLS,
    tool_choice="auto",
    temperature=0,
)

print("finish_reason:", completion.choices[0].finish_reason)
print(completion.choices[0].message.content)


finish_reason: stop
The weather in Paris is currently +10°C.


# 4. The loop

Wrap the same steps: call, run every tool the model asked for, append the results, call again. Stop when the message has no tool calls.

`max_steps` is only a guard against a model that keeps asking for tools.


In [51]:
TOOL_FUNCTIONS = {"get_city": get_city, "get_weather": get_weather}


def ask(messages: list, max_steps: int = 5) -> str:
    for _ in range(max_steps):
        print(messages)
        completion = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
        )
        message = completion.choices[0].message
        messages.append(message)
        print(
            completion.choices[0].finish_reason,
            "|",
            message.content or [(c.function.name, c.function.arguments) for c in message.tool_calls],
        )

        if not message.tool_calls:
            return message.content

        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            result = TOOL_FUNCTIONS[call.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

    return "stopped: the model kept asking for tools"


The city is in the question, so one tool is enough.


In [54]:
messages = [{"role": "user", "content": "temperature today?"}, {"role": "user", "content": "Weather today Im user 4?"}]
ask(messages)


[{'role': 'user', 'content': 'temperature today?'}, {'role': 'user', 'content': 'Weather today Im user 4?'}]
tool_calls | [('get_city', '{"user_id":4}')]
[{'role': 'user', 'content': 'temperature today?'}, {'role': 'user', 'content': 'Weather today Im user 4?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='tool_get_city_FInu95MVNGbZD9t1q5GV', function=Function(arguments='{"user_id":4}', name='get_city'), type='function', index=0)], reasoning=None), {'role': 'tool', 'tool_call_id': 'tool_get_city_FInu95MVNGbZD9t1q5GV', 'content': '{"user_id": 4, "user_name": "Emma Chen", "country": "Singapore", "city": "Singapore"}'}]
tool_calls | [('get_weather', '{"city":"Singapore"}')]
[{'role': 'user', 'content': 'temperature today?'}, {'role': 'user', 'content': 'Weather today Im user 4?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None

'The temperature in Singapore is currently +29°C.'

No city in the question. `get_city` needs a `user_id` and its description says not to guess one, so the model asks for it and stops.

The second call reuses the same `messages`, so the id the user typed is in the conversation.


In [26]:
messages = [{"role": "user", "content": "What's the weather today?"}]
print(ask(messages))

messages.append({"role": "user", "content": "user 2"})
print(ask(messages))


stop | I need to know your user ID to look up your city. Could you please provide it?
I need to know your user ID to look up your city. Could you please provide it?
tool_calls | [('get_city', '{"user_id":2}')]
tool_calls | [('get_weather', '{"city":"Madrid"}')]
stop | The weather in Madrid is currently +12°C.
The weather in Madrid is currently +12°C.


In [27]:
messages = [{"role": "user", "content": "What's the weather today in Berlin and Madrid?"}]
print(ask(messages))

tool_calls | [('get_weather', '{"city":"Berlin"}'), ('get_weather', '{"city":"Madrid"}')]
stop | The weather today in Berlin is 11°C and in Madrid is 12°C.
The weather today in Berlin is 11°C and in Madrid is 12°C.


# Recap

| Question | Tools the model called |
|---|---|
| "What's the weather in Berlin?" | `get_weather` only |
| "What's the weather today?" | none, it asked for the user_id |
| "user 5" | `get_city`, then `get_weather` |

Nothing in the code chose a tool. The model picked from the `description` in `tools` and the words in the question.
